# Kraken Subaccount Transfer Tool

This notebook transfers USD (or other assets) between Kraken subaccounts.

## ⚠️ SECURITY WARNING

**THIS TOOL EXECUTES REAL FINANCIAL TRANSACTIONS**

- Review all parameters carefully before execution
- This code uses your master account API key with Withdraw permissions
- Always verify balances before and after transfers
- Keep your API keys secure and never commit them to version control
- Use at your own risk - see LICENSE for full disclaimer

## What This Does

1. Check balances on both subaccounts (staging and production)
2. Transfer specified amount from one subaccount to another
3. Verify the transfer completed successfully
4. Display updated balances

## Prerequisites

- Master account API key with **Withdraw** permission enabled
- IIBANs (account identifiers) for all accounts
- Configured `.env` file (see `.env.template`)

## Transfer Routes

This tool supports direct transfers between any accounts:
- Master ↔ Subaccount
- Subaccount ↔ Subaccount (direct, no intermediary needed)

All transfers must be initiated using the **master account API key**.

## Step 1: Import Libraries and Initialize

In [ ]:
# Import required libraries
import pandas as pd
from kraken_tools.env_loader import load_kraken_credentials
from kraken_tools.client import KrakenClient

# Display settings for prettier output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.8f}')

print("✅ Imports successful")

## Step 2: Load Credentials

**IMPORTANT:** Update these account names to match your `.env` file.

For example, if your .env has:
- `KRAKEN_API_KEY_MASTER`
- `KRAKEN_IIBAN_STAGING`
- `KRAKEN_IIBAN_PRODUCTION`

Then use:
- `MASTER_ACCOUNT = "MASTER"`
- `STAGING_ACCOUNT = "STAGING"`
- `PRODUCTION_ACCOUNT = "PRODUCTION"`

In [ ]:
# Configure account names (must match suffixes in .env file)
MASTER_ACCOUNT = "MASTER"       # Master account with Withdraw permission
STAGING_ACCOUNT = "STAGING"     # Staging subaccount
PRODUCTION_ACCOUNT = "PRODUCTION"  # Production subaccount

print(f"Master Account: {MASTER_ACCOUNT}")
print(f"Staging Account: {STAGING_ACCOUNT}")
print(f"Production Account: {PRODUCTION_ACCOUNT}")

In [ ]:
# Load credentials from .env file
master_creds = load_kraken_credentials(MASTER_ACCOUNT)
staging_creds = load_kraken_credentials(STAGING_ACCOUNT)
production_creds = load_kraken_credentials(PRODUCTION_ACCOUNT)

print("✅ Credentials loaded successfully")
print(f"\nMaster IIBAN: {master_creds['iiban']}")
print(f"Staging IIBAN: {staging_creds['iiban']}")
print(f"Production IIBAN: {production_creds['iiban']}")

## Step 3: Initialize Kraken Client

**Security Note:** We only use the master account credentials for transfers.
The master API key must have "Withdraw" permission enabled.

In [ ]:
# Initialize master client (required for all transfers)
master_client = KrakenClient(
    api_key=master_creds['api_key'],
    api_secret=master_creds['api_secret']
)

# Initialize subaccount clients (for balance checking only)
staging_client = KrakenClient(
    api_key=staging_creds['api_key'],
    api_secret=staging_creds['api_secret']
)

production_client = KrakenClient(
    api_key=production_creds['api_key'],
    api_secret=production_creds['api_secret']
)

print("\n✅ All clients initialized successfully")

## Step 4: Check Initial Balances

Review current balances before executing any transfers.

In [ ]:
def format_balance(balance_dict):
    """Format balance dictionary as a nice DataFrame."""
    if not balance_dict:
        return pd.DataFrame(columns=['Asset', 'Balance'])
    
    df = pd.DataFrame([
        {'Asset': asset, 'Balance': amount}
        for asset, amount in balance_dict.items()
        if amount > 0  # Only show assets with non-zero balance
    ])
    
    # Handle empty DataFrame (account with zero balance)
    if df.empty:
        return pd.DataFrame(columns=['Asset', 'Balance'])
    
    return df.sort_values('Balance', ascending=False)

# Get balances
print("=" * 60)
print(f"STAGING ACCOUNT BALANCE ({STAGING_ACCOUNT})")
print("=" * 60)
staging_balance = staging_client.get_account_balance()
staging_df = format_balance(staging_balance)
display(staging_df)

print("\n" + "=" * 60)
print(f"PRODUCTION ACCOUNT BALANCE ({PRODUCTION_ACCOUNT})")
print("=" * 60)
production_balance = production_client.get_account_balance()
production_df = format_balance(production_balance)
display(production_df)

## Step 5: Configure Transfer Parameters

**⚠️ REVIEW CAREFULLY BEFORE PROCEEDING**

Set the asset and amount you want to transfer.

In [ ]:
# ============================================================================
# CONFIGURE THESE PARAMETERS
# ============================================================================

TRANSFER_ASSET = "ZUSD"     # Asset to transfer (ZUSD = USD, XXBT = Bitcoin, XETH = Ethereum)
TRANSFER_AMOUNT = 1.0     # Amount to transfer

# Transfer direction: from STAGING to PRODUCTION
FROM_ACCOUNT = staging_creds['iiban']
TO_ACCOUNT = production_creds['iiban']

# ============================================================================

# Display summary
print("Transfer Configuration:")
print("=" * 60)
print(f"Asset: {TRANSFER_ASSET}")
print(f"Amount: {TRANSFER_AMOUNT}")
print(f"From: {FROM_ACCOUNT} (Staging)")
print(f"To: {TO_ACCOUNT} (Production)")
print("=" * 60)

# Check current balances
staging_has = staging_balance.get(TRANSFER_ASSET, 0)
production_has = production_balance.get(TRANSFER_ASSET, 0)

print(f"\nCurrent {TRANSFER_ASSET} Balances:")
print(f"  Staging: {staging_has}")
print(f"  Production: {production_has}")

# Validate sufficient balance
if staging_has < TRANSFER_AMOUNT:
    print(f"\n⚠️ WARNING: Insufficient balance!")
    print(f"   Staging has {staging_has} {TRANSFER_ASSET}, but trying to transfer {TRANSFER_AMOUNT}")
else:
    print(f"\n✅ Sufficient balance available")
    print(f"   After transfer, staging will have: {staging_has - TRANSFER_AMOUNT} {TRANSFER_ASSET}")

## Step 6: Execute Transfer

**⚠️ THIS EXECUTES A REAL FINANCIAL TRANSACTION**

**REVIEW THE PARAMETERS ABOVE BEFORE RUNNING THIS CELL**

You will be prompted to confirm by typing 'yes'.

In [ ]:
print("=" * 60)
print("TRANSFER CONFIRMATION")
print("=" * 60)
print(f"Transfer: {TRANSFER_AMOUNT} {TRANSFER_ASSET}")
print(f"From: {FROM_ACCOUNT} (Staging)")
print(f"To: {TO_ACCOUNT} (Production)")
print("=" * 60)

confirm = input("\nType 'yes' to confirm and execute this transfer: ")

if confirm.lower() == 'yes':
    print("\nExecuting transfer...")
    
    try:
        result = master_client.transfer_between_accounts(
            asset=TRANSFER_ASSET,
            amount=TRANSFER_AMOUNT,
            from_iiban=FROM_ACCOUNT,
            to_iiban=TO_ACCOUNT
        )
        
        print("\n" + "=" * 60)
        print("✅ TRANSFER COMPLETED SUCCESSFULLY")
        print("=" * 60)
        print(f"Transfer ID: {result.get('transfer_id', result.get('refid', 'N/A'))}")
        print(f"Status: {result.get('status', 'Success')}")
        print("\nResult details:")
        for key, value in result.items():
            print(f"  {key}: {value}")
        
    except Exception as e:
        print("\n" + "=" * 60)
        print("❌ TRANSFER FAILED")
        print("=" * 60)
        print(f"Error: {str(e)}")
        print("\nPlease verify:")
        print("  - Master API key has 'Withdraw' permission")
        print("  - Sufficient balance in source account")
        print("  - IIBANs are correct")
        print("  - Asset name is correct (e.g., 'ZUSD' not 'USD')")
else:
    print("\n❌ Transfer cancelled")

## Step 7: Verify Transfer - Check Updated Balances

Run this cell after the transfer to confirm the funds moved correctly.

In [ ]:
print("Refreshing account balances...\n")

# Get updated balances
print("=" * 60)
print(f"UPDATED STAGING BALANCE ({STAGING_ACCOUNT})")
print("=" * 60)
staging_balance_new = staging_client.get_account_balance()
staging_df_new = format_balance(staging_balance_new)
display(staging_df_new)

print("\n" + "=" * 60)
print(f"UPDATED PRODUCTION BALANCE ({PRODUCTION_ACCOUNT})")
print("=" * 60)
production_balance_new = production_client.get_account_balance()
production_df_new = format_balance(production_balance_new)
display(production_df_new)

# Show changes
print("\n" + "=" * 60)
print("BALANCE CHANGES")
print("=" * 60)

staging_old = staging_balance.get(TRANSFER_ASSET, 0)
staging_new = staging_balance_new.get(TRANSFER_ASSET, 0)
staging_change = staging_new - staging_old

production_old = production_balance.get(TRANSFER_ASSET, 0)
production_new = production_balance_new.get(TRANSFER_ASSET, 0)
production_change = production_new - production_old

print(f"{TRANSFER_ASSET} Changes:")
print(f"  Staging: {staging_old} → {staging_new} (change: {staging_change:+.8f})")
print(f"  Production: {production_old} → {production_new} (change: {production_change:+.8f})")

if abs(abs(staging_change) - TRANSFER_AMOUNT) < 0.00001 and abs(production_change - TRANSFER_AMOUNT) < 0.00001:
    print("\n✅ Transfer verified successfully!")
else:
    print("\n⚠️ Balance changes don't match expected transfer amount. Please check Kraken web interface.")

## Complete! ✅

Your transfer has been executed and verified.

### Next Steps

- Verify the transfer in your Kraken web interface
- Check the transfer history in your Kraken account
- Keep this notebook as a record of the transfer

### Security Reminders

- **Never commit your `.env` file** to version control
- Store API keys securely
- Rotate API keys periodically
- Review API key permissions regularly
- Keep this code private if it contains any sensitive information